<a href="https://colab.research.google.com/github/Heng1222/Ohsumed_classification/blob/feat_projection_23D/Model/Projection_23D" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
pip install pandas numpy torch transformers sklearn umap-learn plotly

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [8]:
import pandas as pd
import numpy as np
import json
import torch
import plotly.io as pio
import plotly.express as px
import umap
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import normalize
from sklearn.utils import resample # 用於抽樣

# --- Colab 強制渲染設定 ---
pio.renderers.default = "colab"

# ==========================================
# 0. 實驗參數設定
# ==========================================
ENABLE_COMMON_COMPONENT_REMOVAL = True  # 開關：CCR
MODEL_NAME = "roberta-base"              # 建議換成 PubMedBERT
VISUALIZATION_SAMPLE_SIZE = 2000         # 視覺化抽樣點數 (2000-5000 點是 WebGL 的舒適區)

# ==========================================
# 1. Embedding 模型封裝 ([CLS] Token)
# ==========================================
class CLSEmbeddingModel:
    def __init__(self, model_name=MODEL_NAME):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading {model_name} on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def get_embeddings(self, texts):
        embeddings = []
        batch_size = 16
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch = texts[i : i + batch_size]
                # 確保 truncation=True 避免超長文本導致報錯
                inputs = self.tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(self.device)
                outputs = self.model(**inputs)
                # 取得 [CLS] 向量 (Index 0)
                cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings.extend(cls_embeddings)
        return np.array(embeddings)

# ==========================================
# 2. 數據加載與精準對齊 (C01-C23)
# ==========================================
# 載入類別描述
with open('23D.json', 'r', encoding='utf-8') as f:
    categories_dict = json.load(f)

# 確保對齊 label 0-22 的 C01-C23 順序
cat_keys = [f"C{str(i).zfill(2)}" for i in range(1, 24)]
ordered_descriptions = []
for k in cat_keys:
    full_key = [full for full in categories_dict.keys() if full.startswith(k)][0]
    ordered_descriptions.append(categories_dict[full_key])

# 載入測試資料 (label, abstract)
df = pd.read_csv('ohsumed_dataset.csv') # label: 0-22

embedder = CLSEmbeddingModel()

# 取得原始向量
print(f"Encoding 23 category bases...")
base_vectors = embedder.get_embeddings(ordered_descriptions)  # (23, 768)
print(f"Encoding {len(df)} abstracts (Full Dataset)...")
abstract_embeddings = embedder.get_embeddings(df['abstract'].tolist())  # (N, 768)

# ==========================================
# 3. 第一性原理優化：中心化與正規化
# ==========================================
if ENABLE_COMMON_COMPONENT_REMOVAL:
    print(f">>> Applying Common Component Removal (CCR: True)...")
    # 計算 23 個疾病大類的幾何中心 (醫療廢話廢棄向量)
    global_mean = np.mean(base_vectors, axis=0)
    base_vectors = base_vectors - global_mean
    abstract_embeddings = abstract_embeddings - global_mean
else:
    print(">>> CCR is DISABLED.")

# L2 Normalization -> 讓 Dot Product 等同於 Cosine Similarity
print("Normalizing embeddings for Cosine Similarity...")
base_vectors_norm = normalize(base_vectors, axis=1)
abstract_embeddings_norm = normalize(abstract_embeddings, axis=1)

# ==========================================
# 4. 投影分類與評估 (使用完整資料集)
# ==========================================
# 計算餘弦相似度矩陣 (N, 23)
print("Calculating Projections (Full Dataset)...")
projections = np.dot(abstract_embeddings_norm, base_vectors_norm.T)

# 執行 Argmax 分類：取投影最長者
y_pred = np.argmax(projections, axis=1)
y_true = df['label'].values

print(f"\n--- Performance (Model: {MODEL_NAME}, CCR: {ENABLE_COMMON_COMPONENT_REMOVAL}) ---")
# 加入 zero_division=0 消除分類坍縮時的警告
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(classification_report(y_true, y_pred, target_names=cat_keys, zero_division=0))


Loading roberta-base on cuda...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Encoding 23 category bases...
Encoding 7400 abstracts (Full Dataset)...
>>> Applying Common Component Removal (CCR: True)...
Normalizing embeddings for Cosine Similarity...
Calculating Projections (Full Dataset)...

--- Performance (Model: roberta-base, CCR: True) ---
Macro F1 Score: 0.0917
              precision    recall  f1-score   support

         C01       0.50      0.02      0.04       216
         C02       0.09      0.04      0.06        75
         C03       0.24      0.14      0.18        50
         C04       0.41      0.32      0.36      1030
         C05       0.00      0.00      0.00       223
         C06       0.67      0.03      0.06       354
         C07       0.13      0.30      0.18        63
         C08       0.18      0.10      0.13       250
         C09       0.00      0.00      0.00        63
         C10       1.00      0.00      0.00       557
         C11       0.71      0.04      0.08       125
         C12       0.26      0.07      0.11       342
     

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.




>>> Sampling 2000 points for 3D visualization to optimize rendering speed.

>>> Outputting visualizations directly in Colab cell output...


In [11]:

# ==========================================
# 5. UMAP 3D 視覺化 (加入抽樣邏輯)
# ==========================================
print(f"Reducing dimensions for 3D plot (UMAP + Cosine Metric)...")
# UMAP 參數：metric='cosine' 很重要，因為我們是用角度做分類
reducer = umap.UMAP(n_components=3, random_state=42, metric='cosine')
umap_3d = reducer.fit_transform(projections)

# 建立用於畫圖的完整 DataFrame
plot_df = pd.DataFrame({
    'x': umap_3d[:, 0], 'y': umap_3d[:, 1], 'z': umap_3d[:, 2],
    'Ground Truth': [cat_keys[i] for i in y_true],
    'Prediction': [cat_keys[i] for i in y_pred]
})

# --- 關鍵修正：抽樣以利 Colab 顯示 ---
if len(plot_df) > VISUALIZATION_SAMPLE_SIZE:
    print(f"\n>>> Sampling {VISUALIZATION_SAMPLE_SIZE} points for 3D visualization to optimize rendering speed.")
    # 使用 stratified sampling 確保冷門疾病也能被抽到一些
    try:
        plot_df_sampled = resample(plot_df, replace=False, n_samples=VISUALIZATION_SAMPLE_SIZE, random_state=42, stratify=plot_df['Ground Truth'])
    except ValueError:
        # 如果某些類別樣本太少無法進行分層抽樣，就退化到一般隨機抽樣
        print(" Stratification failed, falling back to simple random sampling.")
        plot_df_sampled = resample(plot_df, replace=False, n_samples=VISUALIZATION_SAMPLE_SIZE, random_state=42)
else:
    plot_df_sampled = plot_df

# === 修改部分：強制指定 Colab 渲染並優化顯示參數 ===
print("\n>>> Outputting visualizations directly in Colab cell output...")

# 畫圖 1: 真實標籤
fig1 = px.scatter_3d(plot_df_sampled, x='x', y='y', z='z', color='Ground Truth',
                     title=f'Ground Truth (CCR: {ENABLE_COMMON_COMPONENT_REMOVAL})',
                     opacity=0.7)
# 優化：設定點的大小、設定顯示器為 colab
fig1.update_traces(marker=dict(size=3))
fig1.show(renderer="colab")

# 畫圖 2: 投影分類預測結果
fig2 = px.scatter_3d(plot_df_sampled, x='x', y='y', z='z', color='Prediction',
                     title=f'Zero-Shot Projection Results (CCR: {ENABLE_COMMON_COMPONENT_REMOVAL})',
                     opacity=0.7)
fig2.update_traces(marker=dict(size=3))
fig2.show(renderer="colab")

Reducing dimensions for 3D plot (UMAP + Cosine Metric)...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.




>>> Sampling 2000 points for 3D visualization to optimize rendering speed.

>>> Outputting visualizations directly in Colab cell output...
